<a href="https://colab.research.google.com/github/anc238github/mycodes/blob/main/website_summariser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code for website summariser

In [22]:
from bs4 import BeautifulSoup
import requests
from openai import OpenAI
from IPython.display import display,Markdown

In [21]:
openai =OpenAI(api_key= "put_the_fucking_key ")

In [23]:
system_prompt = """
You are a funny and sarcastic assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [24]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [25]:
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

In [26]:
def messages_for(website):
  return [
      {'role':'system','content':system_prompt},
          {'role':'user','content':user_prompt_prefix+ website}
      ]

In [27]:
def summarize(url):
  website =fetch_website_contents(url)
  response=openai.chat.completions.create( model= "gpt-4.1-mini", messages=messages_for(website)
  )
  return response.choices[0].message.content

In [28]:
def display_summary(url):
  summary = summarize(url)
  display(Markdown(summary))

In [30]:
summarize("https://cms.co.in")

"# CMS Computers: Because Managing IT, Energy, and Traffic is Apparently Not Enough Fun\n\nWelcome to the exciting world of CMS Computers, where IT & analytics services collide with energy consulting, e-governance, and... traffic solutions? Yep, apparently they do it all—from banking to Bollywood, utilities to transport, and even government services because why not?\n\nThey’ve got products, AWS services (because who doesn’t), and enough buzzwords to make a tech startup jealous: systems integration, software services, deployment solutions, and more. Their client list includes heavy hitters like Bharat Petroleum and Neyveli Lignite Corporation, so they're serious folks.\n\nNews? The hottest scoop is from 2016 about their thrilling participation in some traffic shindig (sing it with me, “Traffic... Traffic...”) and the always heartwarming CSR initiatives where they try to save the world one community at a time. Oh, and they incubate startups — because apparently, CMS is also nurturing fut